In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('data/mushrooms.csv')
df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['class'] = le.fit_transform(df['class'])
df.head()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder()
X_cols = df.columns[1:]
df[X_cols] = oe.fit_transform(df[X_cols])
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop(['class'], axis=1)
y = df['class']
X_train, X_test, y_train, y_test = train_test_split(X,
                                                   y,
                                                   test_size=0.3,
                                                   shuffle=True,
                                                   stratify=y,
                                                   random_state=42)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif

gbc = GradientBoostingClassifier(max_depth=5, random_state=42)

f1_score_list = []

for k in range(1, 14):
    selector = SelectKBest(mutual_info_classif, k=k)
    selector.fit(X_train, y_train)
    
    sel_X_train = selector.transform(X_train)
    sel_X_test = selector.transform(X_test)
    
    gbc.fit(sel_X_train, y_train)
    kbest_preds = gbc.predict(sel_X_test)
    
    f1_score_kbest = round(f1_score(y_test, kbest_preds, average='weighted'), 3)
    
    f1_score_list.append(f1_score_kbest)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()

x = np.arange(1, 14)
y = f1_score_list

ax.bar(x, y, width=0.2)
ax.set_xlabel('Number of features selected using mutual information')
ax.set_ylabel('F1-Score (weighted)')
ax.set_ylim(0, 1.2)
ax.set_xticks(np.arange(1, 14))
ax.set_xticklabels(np.arange(1, 14), fontsize=12)

for i, v in enumerate(y):
    plt.text(x=i+1, y=v+0.05, s=str(v), ha='center')
    
plt.tight_layout()

In [ ]:
selector = SelectKBest(mutual_info_classif, k=9)
selector.fit(X_train, y_train)

selected_features_mask = selector.get_support()

selected_features = X_train.columns[selected_features_mask]

selected_features